In [3]:
# ============================================================
# TASK 3: DATA-DRIVEN FX STRATEGY
# Linear, Ridge/Lasso, Random Forest
# Look-ahead safe: train <= 2019-12-31, test >= 2020-01-01
# ============================================================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score

# ------------------------------------------------------------
# 1. Load workbook
# ------------------------------------------------------------

FILE_PATH = "DataTables Updated thru 15May25.xlsx"

xls = pd.ExcelFile(FILE_PATH)

print("Available sheets:")
print(xls.sheet_names)

sheets_to_use = [
    "FX SPOT",
    "ATM VOLS",
    "3M 25D RR",
    "3M DEPOSIT RATES",
    "10Y YIELD",
    "EQUITY INDICES",
    "COMDTY",
    "CREDIT SPREADS",
    "JPM EASI",
    "IMM POSITIONING",
    "1M EUR FWD"
]

dfs = []

for sheet in sheets_to_use:
    if sheet not in xls.sheet_names:
        print(f"Skipping missing sheet: {sheet}")
        continue

    df = pd.read_excel(FILE_PATH, sheet_name=sheet)

    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"])
    df = df.set_index("Date")
    df = df.sort_index()

    df = df.apply(pd.to_numeric, errors="coerce")

    # Remove empty columns
    df = df.dropna(axis=1, how="all")

    # Prefix column names
    df.columns = [f"{sheet}__{c}" for c in df.columns]

    dfs.append(df)

data = pd.concat(dfs, axis=1)
data = data.sort_index()
data = data[~data.index.duplicated(keep="first")]

# Forward-fill only using past information
data = data.ffill()

print("\nRaw combined data:")
print(data.shape)
print(data.index.min(), "to", data.index.max())

# ------------------------------------------------------------
# 2. Select target variable
# ------------------------------------------------------------

eurusd_candidates = [
    c for c in data.columns
    if "EUR" in c.upper() and "USD" in c.upper() and "FX SPOT" in c.upper()
]

print("\nEURUSD candidates:")
print(eurusd_candidates)

if len(eurusd_candidates) == 0:
    raise ValueError("No EURUSD FX spot column found. Inspect FX SPOT sheet column names.")

target_price_col = eurusd_candidates[0]

print("\nUsing target column:")
print(target_price_col)

data["target"] = np.log(data[target_price_col]).diff().shift(-1)

# ------------------------------------------------------------
# 3. Feature engineering
# ------------------------------------------------------------
# All features use only information available at time t.
# Final features are lagged by 1 day before modelling.

features = pd.DataFrame(index=data.index)

for col in data.columns:
    if col == "target":
        continue

    s = data[col].copy()

    # Level
    features[f"{col}_level"] = s

    # Short-term change
    features[f"{col}_chg_1d"] = s.diff(1)

    # Weekly change
    features[f"{col}_chg_5d"] = s.diff(5)

    # Monthly change / momentum
    features[f"{col}_chg_21d"] = s.diff(21)

    # Log-return features only where strictly positive
    if (s > 0).sum() > 100:
        lr = np.log(s).diff()
        features[f"{col}_logret_1d"] = lr
        features[f"{col}_mom_21d"] = np.log(s).diff(21)
        features[f"{col}_vol_21d"] = lr.rolling(21).std()

# ------------------------------------------------------------
# 4. Missing-data control
# ------------------------------------------------------------

# Remove columns with excessive missing values
features = features.loc[:, features.isna().mean() < 0.40]

# Lag all features to prevent look-ahead bias
features = features.shift(1)

dataset = pd.concat([features, data["target"]], axis=1)
dataset = dataset.replace([np.inf, -np.inf], np.nan)

# Drop rows only after removing bad feature columns
dataset = dataset.dropna()

print("\nFinal modelling dataset:")
print(dataset.shape)
print(dataset.index.min(), "to", dataset.index.max())

if dataset.empty:
    raise ValueError(
        "Dataset is still empty after preprocessing. "
        "Reduce missing-value threshold or inspect sheet/date alignment."
    )

# ------------------------------------------------------------
# 5. Train/test split
# ------------------------------------------------------------

X = dataset.drop(columns=["target"])
y = dataset["target"]

train_end = "2019-12-31"
test_start = "2020-01-01"

X_train = X.loc[:train_end]
y_train = y.loc[:train_end]

X_test = X.loc[test_start:]
y_test = y.loc[test_start:]

print("\nTrain/test split:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

if X_train.empty:
    raise ValueError("Training set is empty. Check date range and dataset start date.")

if X_test.empty:
    raise ValueError("Test set is empty. Check date range and dataset end date.")

# ------------------------------------------------------------
# 6. Models
# ------------------------------------------------------------

tscv = TimeSeriesSplit(n_splits=5)

linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RidgeCV(
        alphas=np.logspace(-4, 4, 50),
        cv=tscv
    ))
])

lasso_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(
        alphas=np.logspace(-5, 1, 50),
        cv=tscv,
        max_iter=30000,
        random_state=42
    ))
])

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=5,
    min_samples_leaf=20,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

models = {
    "Linear Regression": linear_model,
    "Ridge": ridge_model,
    "Lasso": lasso_model,
    "Random Forest": rf_model
}

# ------------------------------------------------------------
# 7. Strategy evaluation function
# ------------------------------------------------------------

def evaluate_strategy(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)

    pred = pd.Series(
        model.predict(X_test),
        index=X_test.index,
        name=name
    )

    signal = np.sign(pred)

    strategy_return = signal * y_test
    strategy_return.name = name

    ann_return = strategy_return.mean() * 252
    ann_vol = strategy_return.std() * np.sqrt(252)

    if ann_vol == 0 or np.isnan(ann_vol):
        sharpe = np.nan
    else:
        sharpe = ann_return / ann_vol

    result = {
        "Model": name,
        "Test MSE": mean_squared_error(y_test, pred),
        "Test R2": r2_score(y_test, pred),
        "Annual Return": ann_return,
        "Annual Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Hit Rate": (np.sign(pred) == np.sign(y_test)).mean(),
        "Average Daily Turnover": signal.diff().abs().fillna(0).mean()
    }

    return result, pred, strategy_return

# ------------------------------------------------------------
# 8. Run all models
# ------------------------------------------------------------

all_results = []
predictions = {}
strategy_returns = {}

for name, model in models.items():
    print(f"Running {name}...")

    result, pred, strat_ret = evaluate_strategy(
        name, model,
        X_train, y_train,
        X_test, y_test
    )

    all_results.append(result)
    predictions[name] = pred
    strategy_returns[name] = strat_ret

results_table = pd.DataFrame(all_results)
results_table = results_table.sort_values("Sharpe Ratio", ascending=False)

print("\nMODEL COMPARISON")
print(results_table)

# ------------------------------------------------------------
# 9. Best regularised model: Ridge vs Lasso
# ------------------------------------------------------------

ridge_lasso = results_table[results_table["Model"].isin(["Ridge", "Lasso"])]
best_regularised_model = ridge_lasso.iloc[0]["Model"]

print("\nBest regularised model:")
print(best_regularised_model)

# ------------------------------------------------------------
# 10. Equity curves
# ------------------------------------------------------------

strategy_returns_df = pd.DataFrame(strategy_returns)
predictions_df = pd.DataFrame(predictions)

equity_curves = (1 + strategy_returns_df).cumprod()

print("\nFinal cumulative performance:")
print(equity_curves.iloc[-1].sort_values(ascending=False))

# ------------------------------------------------------------
# 11. Random Forest feature importance
# ------------------------------------------------------------

rf_model.fit(X_train, y_train)

rf_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nTop 20 Random Forest features:")
print(rf_importance.head(20))

# ------------------------------------------------------------
# 12. Linear / Ridge / Lasso coefficients
# ------------------------------------------------------------

coefficient_tables = {}

for model_name in ["Linear Regression", "Ridge", "Lasso"]:
    fitted_model = models[model_name]
    coefs = fitted_model.named_steps["model"].coef_

    coef_table = pd.DataFrame({
        "Feature": X_train.columns,
        "Coefficient": coefs
    })

    coef_table["Abs Coefficient"] = coef_table["Coefficient"].abs()
    coef_table = coef_table.sort_values("Abs Coefficient", ascending=False)

    coefficient_tables[model_name] = coef_table

    print(f"\nTop 20 coefficients: {model_name}")
    print(coef_table.head(20))

# ------------------------------------------------------------
# 13. Save outputs
# ------------------------------------------------------------

output_file = "Task3_ML_Strategy_Results.xlsx"

with pd.ExcelWriter(output_file) as writer:
    results_table.to_excel(writer, sheet_name="Model Comparison", index=False)
    equity_curves.to_excel(writer, sheet_name="Equity Curves")
    predictions_df.to_excel(writer, sheet_name="Predictions")
    strategy_returns_df.to_excel(writer, sheet_name="Strategy Returns")
    rf_importance.to_excel(writer, sheet_name="RF Importance", index=False)

    for model_name, coef_table in coefficient_tables.items():
        sheet_name = model_name[:31]
        coef_table.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\nSaved results to: {output_file}")

Available sheets:
['VARIABLES', 'FX SPOT', 'ATM VOLS', '3M 25D RR', '3M DEPOSIT RATES', '10Y YIELD', 'EQUITY INDICES', 'COMDTY', 'CREDIT SPREADS', 'JPM EASI', 'IMM POSITIONING', '1M EUR FWD', 'BBG Formula']

Raw combined data:
(11510, 135)
1993-11-10 00:00:00 to 2025-05-15 00:00:00

EURUSD candidates:
['FX SPOT__EURUSD CURNCY']

Using target column:
FX SPOT__EURUSD CURNCY


/var/folders/dl/c7mgwnp933g33481293_fk9m0000gn/T/ipykernel_3783/4145390495.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"{col}_chg_5d"] = s.diff(5)
/var/folders/dl/c7mgwnp933g33481293_fk9m0000gn/T/ipykernel_3783/4145390495.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"{col}_chg_21d"] = s.diff(21)
/var/folders/dl/c7mgwnp933g33481293_fk9m0000gn/T/ipykernel_3783/4145390495.py:129: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many time


Final modelling dataset:
(1726, 802)
2005-12-25 00:00:00 to 2025-05-14 00:00:00

Train/test split:
X_train: (1450, 801)
X_test: (276, 801)
Running Linear Regression...
Running Ridge...
Running Lasso...
Running Random Forest...

MODEL COMPARISON
               Model  Test MSE        Test R2  Annual Return  \
1              Ridge  0.000021      -0.136194       0.067444   
0  Linear Regression  9.517006 -521473.289346       0.065407   
2              Lasso  0.000018      -0.004673      -0.065407   
3      Random Forest  0.000019      -0.015588      -0.135568   

   Annual Volatility  Sharpe Ratio  Hit Rate  Average Daily Turnover  
1           0.067932      0.992823  0.347826                0.043478  
0           0.067939      0.962722  0.347826                0.000000  
2           0.067939     -0.962722  0.358696                0.000000  
3           0.067525     -2.007674  0.311594                0.398551  

Best regularised model:
Ridge

Final cumulative performance:
Ridge           

In [6]:
"""
Task 3 Ridge model: predictive signals for Gold, Silver, Brent Crude and Natural Gas.

This script is designed to sit next to the existing neural-network code block.
It deliberately mirrors the NN output structure so the existing portfolio construction
block can be reused with only the signal-file path changed.

Inputs expected
---------------
1. commodity_daily_returns.csv
   From Task 0 output. Must contain daily commodity returns with columns such as:
   GOLD 100 OZ, SILVER, BRENT CRUDE, NATURAL GAS.

2. DataTables Updated thru 15May25.xlsx
   Auxiliary market variables workbook.

Output folder
-------------
task3_ridge_outputs/
    ridge_test_predictions_long.csv
    ridge_test_signals_wide.csv
    ridge_test_predicted_returns_wide.csv
    ridge_model_metrics.csv
    ridge_selected_features_by_commodity.json

Compatibility copies are also saved with NN-style names in the same folder:
    nn_test_predictions_long.csv
    nn_test_signals_wide.csv
    nn_test_predicted_returns_wide.csv
    nn_model_metrics.csv

So, to reuse the existing portfolio construction block, set:
    SIGNALS_FILE = Path("task3_ridge_outputs/nn_test_signals_wide.csv")
"""

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# =============================================================================
# User-editable settings
# =============================================================================
COMMODITY_RETURNS_FILE = Path("commodity_daily_returns.csv")
AUX_FILE = Path("DataTables Updated thru 15May25.xlsx")
OUTPUT_DIR = Path("task3_ridge_outputs")

DATE_COL = "Date"
TRAIN_END = pd.Timestamp("2019-12-31")
TEST_START = pd.Timestamp("2020-01-01")
MODEL_START_DATE = pd.Timestamp("2003-11-01")

TRADING_DAYS = 252
TARGET_HORIZON_DAYS = 21
FEATURE_LAG_DAYS = 1
VOL_LOOKBACK_DAYS = 60
MIN_FEATURE_COVERAGE_IN_TRAIN = 0.45
TARGET_CLIP_ABS = 5.0

# Because the target uses a forward 21-trading-day return, the training/CV sample
# must stop 21 business days before TRAIN_END. Otherwise late-December 2019 labels
# would include January 2020 returns.
SAFE_TRAIN_END = TRAIN_END - pd.tseries.offsets.BDay(TARGET_HORIZON_DAYS)

RIDGE_ALPHAS = np.logspace(-4, 4, 80)

SELECTED_COMMODITIES = {
    "Gold": "GOLD 100 OZ",
    "Silver": "SILVER",
    "Brent_Crude": "BRENT CRUDE",
    "Natural_Gas": "NATURAL GAS",
}

AUX_SHEETS_TO_USE = [
    "FX SPOT",
    "ATM VOLS",
    "3M 25D RR",
    "3M DEPOSIT RATES",
    "10Y YIELD",
    "EQUITY INDICES",
    "COMDTY",
    "CREDIT SPREADS",
    "JPM EASI",
    "IMM POSITIONING",
    "1M EUR FWD",
]


# =============================================================================
# Helpers
# =============================================================================
def find_existing_path(path: Path) -> Path:
    """Search current folder and /mnt/data for a file."""
    candidates = [
        Path(path),
        Path.cwd() / path,
        Path.cwd() / path.name,
        Path("/mnt/data") / path,
        Path("/mnt/data") / path.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {path}. Tried: {candidates}")


def clean_numeric_sheet(df: pd.DataFrame) -> pd.DataFrame:
    """Convert a raw Excel worksheet into a date-indexed numeric DataFrame."""
    out = df.copy()
    first_col = out.columns[0]
    out = out.rename(columns={first_col: "Date"})
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out = out.dropna(subset=["Date"])
    out = out.sort_values("Date")
    out = out.drop_duplicates(subset=["Date"], keep="last")
    out = out.set_index("Date")
    out = out.apply(pd.to_numeric, errors="coerce")
    out = out.dropna(axis=1, how="all")
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def pct_change(df_or_s, periods: int):
    return df_or_s.pct_change(periods=periods, fill_method=None).replace([np.inf, -np.inf], np.nan)


def log_diff(s: pd.Series, periods: int = 1) -> pd.Series:
    return np.log(s.where(s > 0)).diff(periods).replace([np.inf, -np.inf], np.nan)


def basket_mean(df: pd.DataFrame, columns: Iterable[str], min_count: int = 1) -> pd.Series:
    cols = [c for c in columns if c in df.columns]
    if len(cols) == 0:
        return pd.Series(index=df.index, dtype=float)
    valid_count = df[cols].notna().sum(axis=1)
    out = df[cols].mean(axis=1, skipna=True)
    return out.where(valid_count >= min_count)


def compound_return_from_daily_returns(r: pd.Series, window: int) -> pd.Series:
    """Trailing compounded return over `window` trading days."""
    return (1.0 + r).rolling(window, min_periods=max(2, window // 2)).apply(np.prod, raw=True) - 1.0


def forward_return_from_daily_returns(r: pd.Series, horizon: int) -> pd.Series:
    """
    Forward compounded return using returns from t+1 to t+horizon.
    This matches a signal formed using information up to t and held forward.
    In this pipeline features are lagged by one day, so row t is conservative.
    """
    return (1.0 + r).shift(-1).rolling(horizon, min_periods=horizon).apply(np.prod, raw=True).shift(-(horizon - 1)) - 1.0


def annualised_vol(r: pd.Series, window: int = VOL_LOOKBACK_DAYS) -> pd.Series:
    return r.rolling(window, min_periods=max(20, window // 2)).std() * math.sqrt(TRADING_DAYS)


# =============================================================================
# Load commodities and auxiliary data
# =============================================================================
def load_commodity_returns(file_path: Path = COMMODITY_RETURNS_FILE) -> pd.DataFrame:
    file_path = find_existing_path(file_path)
    returns = pd.read_csv(file_path, index_col=0, parse_dates=True)
    returns.index = pd.to_datetime(returns.index)
    returns = returns.sort_index()
    returns.columns = returns.columns.astype(str).str.strip()
    returns = returns.apply(pd.to_numeric, errors="coerce")
    returns = returns.replace([np.inf, -np.inf], np.nan)
    return returns


def load_auxiliary_workbook(file_path: Path = AUX_FILE) -> Dict[str, pd.DataFrame]:
    file_path = find_existing_path(file_path)
    raw_sheets = pd.read_excel(file_path, sheet_name=AUX_SHEETS_TO_USE)
    aux = {}
    for sheet_name, raw_df in raw_sheets.items():
        try:
            cleaned = clean_numeric_sheet(raw_df)
            cleaned.columns = [f"{sheet_name}__{c}" for c in cleaned.columns]
            aux[sheet_name] = cleaned
        except Exception as exc:
            print(f"Warning: skipped sheet {sheet_name}: {exc}")
    return aux


def combine_auxiliary_sheets(aux: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    parts = [df for df in aux.values() if df.shape[1] > 0]
    if len(parts) == 0:
        raise ValueError("No usable auxiliary sheets were loaded.")
    data = pd.concat(parts, axis=1).sort_index()
    data = data[~data.index.duplicated(keep="last")]
    data = data.replace([np.inf, -np.inf], np.nan)
    # Forward fill short gaps only. Longer gaps remain NaN and are handled by the imputer later.
    data = data.ffill(limit=5)
    return data


# =============================================================================
# Feature construction
# =============================================================================
def first_or_nan(data: pd.DataFrame, columns: Iterable[str]) -> pd.Series:
    cols = [c for c in columns if c in data.columns]
    if len(cols) == 0:
        return pd.Series(index=data.index, dtype=float)
    return data[cols[0]]


def build_common_auxiliary_features(data: pd.DataFrame) -> pd.DataFrame:
    """Economically motivated auxiliary features shared by all four commodities."""
    common_features = pd.DataFrame(index=data.index)

    # FX spot and USD strength
    fx_spot_cols = [c for c in data.columns if c.upper().startswith("FX SPOT__")]
    usd_strength_components = []
    commodity_fx_components = []
    em_fx_stress_components = []

    for col in fx_spot_cols:
        s = data[col]
        if (s > 0).sum() < 100:
            continue
        ret_21d = log_diff(s, 21)
        name = col.upper()

        # For common XXXUSD quotes, lower pair = stronger USD. For USDXXX, higher pair = stronger USD.
        if any(pair in name for pair in ["EURUSD", "GBPUSD", "AUDUSD", "CADUSD", "NZDUSD"]):
            usd_component = -ret_21d
        elif "USD" in name:
            usd_component = ret_21d
        else:
            continue

        usd_strength_components.append(usd_component)

        if any(pair in name for pair in ["AUDUSD", "CADUSD"]):
            commodity_fx_components.append(ret_21d)
        elif any(pair in name for pair in ["USDNOK", "USDMXN", "USDBRL", "USDZAR"]):
            commodity_fx_components.append(-ret_21d)

        if any(pair in name for pair in ["USDMXN", "USDBRL", "USDZAR", "USDTRY", "USDINR", "USDKRW"]):
            em_fx_stress_components.append(ret_21d)

    if usd_strength_components:
        common_features["usd_strength_ret_21d"] = pd.concat(usd_strength_components, axis=1).mean(axis=1)
    if commodity_fx_components:
        common_features["commodity_fx_strength_ret_21d"] = pd.concat(commodity_fx_components, axis=1).mean(axis=1)
    if em_fx_stress_components:
        common_features["em_fx_stress_ret_21d"] = pd.concat(em_fx_stress_components, axis=1).mean(axis=1)

    # FX implied volatility baskets
    fx_vol_cols = [c for c in data.columns if c.upper().startswith("ATM VOLS__")]
    fx_vol_basket = basket_mean(data, fx_vol_cols, min_count=3)
    common_features["fx_atm_vol_level"] = fx_vol_basket
    common_features["fx_atm_vol_chg_21d"] = fx_vol_basket.diff(21)

    # FX risk reversal / skew
    rr_cols = [c for c in data.columns if c.upper().startswith("3M 25D RR__")]
    rr_basket = basket_mean(data, rr_cols, min_count=3)
    common_features["fx_rr_level"] = rr_basket
    common_features["fx_rr_chg_21d"] = rr_basket.diff(21)

    # Rates and curve proxy
    yield_10y_cols = [c for c in data.columns if c.upper().startswith("10Y YIELD__")]
    us10_cols = [c for c in yield_10y_cols if any(x in c.upper() for x in ["USGG10", "US", "TREASURY"])]
    us10 = first_or_nan(data, us10_cols)
    global_10y = basket_mean(data, yield_10y_cols, min_count=3)
    common_features["us_10y_level"] = us10
    common_features["us_10y_chg_21d"] = us10.diff(21)
    common_features["global_10y_level"] = global_10y
    common_features["global_10y_chg_21d"] = global_10y.diff(21)

    deposit_3m_cols = [c for c in data.columns if c.upper().startswith("3M DEPOSIT RATES__")]
    us3m_cols = [c for c in deposit_3m_cols if any(x in c.upper() for x in ["USD", "US"])]
    us3m = first_or_nan(data, us3m_cols)
    common_features["us_3m_rate_level"] = us3m
    common_features["us_curve_10y_minus_3m"] = us10 - us3m

    # Equity risk appetite
    equity_cols = [c for c in data.columns if c.upper().startswith("EQUITY INDICES__")]
    equity_ret_cols = [c for c in equity_cols if (data[c] > 0).sum() > 100]
    equity_rets = pd.DataFrame({c: log_diff(data[c], 21) for c in equity_ret_cols}, index=data.index)
    common_features["global_equity_ret_21d"] = equity_rets.mean(axis=1, skipna=True)

    spx_cols = [c for c in equity_cols if any(x in c.upper() for x in ["SPX", "S&P", "SP500"])]
    spx = first_or_nan(data, spx_cols)
    spx_ret = log_diff(spx, 1)
    common_features["spx_realised_vol_21d"] = spx_ret.rolling(21, min_periods=15).std() * math.sqrt(TRADING_DAYS)
    common_features["spx_realised_vol_60d"] = spx_ret.rolling(60, min_periods=40).std() * math.sqrt(TRADING_DAYS)

    # Credit spreads
    credit_cols = [c for c in data.columns if c.upper().startswith("CREDIT SPREADS__")]
    hy_cols = [c for c in credit_cols if "HY" in c.upper() or "HIGH YIELD" in c.upper()]
    ig_cols = [c for c in credit_cols if "IG" in c.upper() or "INVESTMENT" in c.upper()]
    hy = first_or_nan(data, hy_cols)
    ig = first_or_nan(data, ig_cols)
    common_features["credit_hy_level"] = hy
    common_features["credit_hy_chg_21d"] = hy.diff(21)
    common_features["credit_ig_level"] = ig
    common_features["credit_ig_chg_21d"] = ig.diff(21)
    common_features["credit_hy_minus_ig"] = hy - ig

    # JPM EASI
    easi_cols = [c for c in data.columns if c.upper().startswith("JPM EASI__")]
    easi_basket = basket_mean(data, easi_cols, min_count=2)
    common_features["easi_global_level"] = easi_basket
    common_features["easi_global_chg_21d"] = easi_basket.diff(21)

    # IMM positioning
    imm_cols = [c for c in data.columns if c.upper().startswith("IMM POSITIONING__")]
    imm_basket = basket_mean(data, imm_cols, min_count=2)
    common_features["imm_positioning_level"] = imm_basket
    common_features["imm_positioning_chg_21d"] = imm_basket.diff(21)

    # 1M EUR forward
    eur_fwd_cols = [c for c in data.columns if c.upper().startswith("1M EUR FWD__")]
    eur_fwd = first_or_nan(data, eur_fwd_cols)
    common_features["eur_1m_forward_level"] = eur_fwd
    common_features["eur_1m_forward_chg_21d"] = eur_fwd.diff(21)

    # Auxiliary commodity proxies if present, for example XAU and CO1 in the auxiliary workbook.
    aux_comdty_cols = [c for c in data.columns if c.upper().startswith("COMDTY__")]
    for col in aux_comdty_cols:
        short_name = col.replace("COMDTY__", "aux_").replace(" ", "_").replace("/", "_")
        if (data[col] > 0).sum() > 100:
            common_features[f"{short_name}_ret_21d"] = log_diff(data[col], 21)
            common_features[f"{short_name}_vol_21d"] = log_diff(data[col], 1).rolling(21, min_periods=15).std() * math.sqrt(TRADING_DAYS)

    common_features = common_features.replace([np.inf, -np.inf], np.nan)
    common_features = common_features.dropna(axis=1, how="all")
    return common_features


def build_commodity_features(
    commodity_label: str,
    commodity_col: str,
    returns: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.Series]:
    """Build commodity-specific features and the 21-day forward target."""
    r = returns[commodity_col]
    synthetic_level = (1.0 + r.fillna(0.0)).cumprod()
    synthetic_level = synthetic_level.where(r.notna().cummax())

    features = pd.DataFrame(index=returns.index)
    features["own_ret_5d"] = compound_return_from_daily_returns(r, 5)
    features["own_ret_21d"] = compound_return_from_daily_returns(r, 21)
    features["own_ret_63d"] = compound_return_from_daily_returns(r, 63)
    features["own_ret_252d"] = compound_return_from_daily_returns(r, 252)
    features["own_realised_vol_21d"] = annualised_vol(r, 21)
    features["own_realised_vol_60d"] = annualised_vol(r, 60)
    features["own_vol_ratio_21d_60d"] = features["own_realised_vol_21d"] / features["own_realised_vol_60d"]
    features["own_drawdown_from_63d_high"] = synthetic_level / synthetic_level.rolling(63, min_periods=21).max() - 1.0
    features["own_drawdown_from_252d_high"] = synthetic_level / synthetic_level.rolling(252, min_periods=126).max() - 1.0

    market_factor = returns.mean(axis=1, skipna=True)
    features["commodity_market_ret_21d"] = compound_return_from_daily_returns(market_factor, 21)
    features["commodity_market_ret_63d"] = compound_return_from_daily_returns(market_factor, 63)
    features["commodity_market_vol_60d"] = annualised_vol(market_factor, 60)

    forward_21d_return = forward_return_from_daily_returns(r, TARGET_HORIZON_DAYS)
    realised_vol_60d = annualised_vol(r, VOL_LOOKBACK_DAYS)
    target = (forward_21d_return / realised_vol_60d).replace([np.inf, -np.inf], np.nan)
    target.name = "target_vol_scaled_21d_return"

    return features.replace([np.inf, -np.inf], np.nan), target


def make_model_panel(
    commodity_label: str,
    commodity_col: str,
    returns: pd.DataFrame,
    common_features: pd.DataFrame,
) -> pd.DataFrame:
    commodity_features, target = build_commodity_features(commodity_label, commodity_col, returns)

    # Align auxiliary features to the commodity return calendar.
    aux_aligned = common_features.reindex(returns.index).ffill(limit=5)

    X_raw = pd.concat([commodity_features, aux_aligned], axis=1)
    X = X_raw.shift(FEATURE_LAG_DAYS)

    panel = pd.concat([X, target], axis=1)
    panel["commodity_label"] = commodity_label
    panel["commodity_name"] = commodity_col
    panel = panel.loc[panel.index >= MODEL_START_DATE]
    panel = panel.replace([np.inf, -np.inf], np.nan)
    return panel


# =============================================================================
# Ridge fitting
# =============================================================================
def train_test_split_panel(panel: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    feature_cols = [c for c in panel.columns if c not in ["target_vol_scaled_21d_return", "commodity_label", "commodity_name"]]

    train_window = panel.loc[:SAFE_TRAIN_END]
    coverage = train_window[feature_cols].notna().mean()
    selected_features = coverage[coverage >= MIN_FEATURE_COVERAGE_IN_TRAIN].index.tolist()

    if len(selected_features) == 0:
        raise ValueError("No features passed the training coverage filter.")

    train = panel.loc[:SAFE_TRAIN_END].dropna(subset=["target_vol_scaled_21d_return"]).copy()
    test = panel.loc[panel.index >= TEST_START].copy()

    return train, test, selected_features


def fit_ridge_for_commodity(
    commodity_label: str,
    commodity_col: str,
    panel: pd.DataFrame,
) -> Tuple[pd.DataFrame, Dict, pd.DataFrame, List[str]]:
    train, test, selected_features = train_test_split_panel(panel)

    if len(train) < 500:
        raise ValueError(f"Insufficient training rows for {commodity_label}: {len(train)}")
    if len(test) == 0:
        raise ValueError(f"No test rows for {commodity_label}.")

    X_train = train[selected_features]
    y_train = train["target_vol_scaled_21d_return"].clip(-TARGET_CLIP_ABS, TARGET_CLIP_ABS)
    X_test = test[selected_features]
    y_test = test["target_vol_scaled_21d_return"]

    n_splits = min(5, max(2, len(X_train) // 500))
    tscv = TimeSeriesSplit(n_splits=n_splits)

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RidgeCV(
            alphas=RIDGE_ALPHAS,
            cv=tscv,
            scoring="neg_mean_squared_error",
        )),
    ])

    model.fit(X_train, y_train)

    pred = pd.Series(
        model.predict(X_test),
        index=X_test.index,
        name="predicted_vol_scaled_21d_return",
    )

    signal_tanh = np.tanh(pred)
    signal_direction = np.where(signal_tanh > 0, 1, -1)

    prediction_df = pd.DataFrame({
        "Dates": X_test.index,
        "commodity_label": commodity_label,
        "commodity_name": commodity_col,
        "predicted_vol_scaled_21d_return": pred.values,
        "signal_tanh": signal_tanh.values,
        "signal_direction": signal_direction,
        "target_vol_scaled_21d_return": y_test.values,
    })

    eval_df = prediction_df.dropna(subset=["target_vol_scaled_21d_return"]).copy()
    if len(eval_df) > 10:
        test_mse = mean_squared_error(eval_df["target_vol_scaled_21d_return"], eval_df["predicted_vol_scaled_21d_return"])
        test_r2 = r2_score(eval_df["target_vol_scaled_21d_return"], eval_df["predicted_vol_scaled_21d_return"])
        test_corr = np.corrcoef(eval_df["target_vol_scaled_21d_return"], eval_df["predicted_vol_scaled_21d_return"])[0, 1]
        directional_accuracy = (np.sign(eval_df["target_vol_scaled_21d_return"]) == np.sign(eval_df["predicted_vol_scaled_21d_return"])).mean()
    else:
        test_mse = np.nan
        test_r2 = np.nan
        test_corr = np.nan
        directional_accuracy = np.nan

    metrics = {
        "commodity_label": commodity_label,
        "commodity_name": commodity_col,
        "model": "RidgeCV",
        "n_train": len(train),
        "n_test_rows": len(test),
        "n_test_eval_rows": len(eval_df),
        "n_features": len(selected_features),
        "selected_alpha": float(model.named_steps["model"].alpha_),
        "safe_train_end": str(SAFE_TRAIN_END.date()),
        "test_mse": test_mse,
        "test_r2": test_r2,
        "test_corr": test_corr,
        "test_directional_accuracy": directional_accuracy,
    }

    ridge_coef = model.named_steps["model"].coef_
    coef_df = pd.DataFrame({
        "commodity_label": commodity_label,
        "commodity_name": commodity_col,
        "feature": selected_features,
        "coefficient": ridge_coef,
    })
    coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
    coef_df = coef_df.sort_values(["commodity_label", "abs_coefficient"], ascending=[True, False])

    return prediction_df, metrics, coef_df, selected_features


# =============================================================================
# Main pipeline
# =============================================================================
def run_task3_ridge_pipeline(
    commodity_returns_file: Path = COMMODITY_RETURNS_FILE,
    aux_file: Path = AUX_FILE,
    output_dir: Path = OUTPUT_DIR,
    selected_commodities: Dict[str, str] = SELECTED_COMMODITIES,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    returns = load_commodity_returns(commodity_returns_file)

    missing = [col for col in selected_commodities.values() if col not in returns.columns]
    if missing:
        raise ValueError(
            "Missing required commodity return columns: " + ", ".join(missing) +
            f"\nAvailable columns include: {list(returns.columns[:20])}"
        )

    aux = load_auxiliary_workbook(aux_file)
    aux_data = combine_auxiliary_sheets(aux)
    common_features = build_common_auxiliary_features(aux_data)

    all_predictions = []
    all_metrics = []
    all_coefs = []
    selected_feature_map = {}

    for label, commodity_col in selected_commodities.items():
        print(f"\nFitting Ridge for {label} ({commodity_col})...")
        panel = make_model_panel(label, commodity_col, returns, common_features)
        pred_df, metrics, coef_df, selected_features = fit_ridge_for_commodity(label, commodity_col, panel)

        all_predictions.append(pred_df)
        all_metrics.append(metrics)
        all_coefs.append(coef_df)
        selected_feature_map[label] = selected_features

        print(
            f"  alpha={metrics['selected_alpha']:.6f}, "
            f"features={metrics['n_features']}, "
            f"train={metrics['n_train']}, "
            f"test_corr={metrics['test_corr']:.4f}, "
            f"directional_accuracy={metrics['test_directional_accuracy']:.3f}"
        )

    predictions_long = pd.concat(all_predictions, axis=0, ignore_index=True)
    metrics_df = pd.DataFrame(all_metrics)
    ridge_coefficients = pd.concat(all_coefs, axis=0, ignore_index=True)

    signal_wide = predictions_long.pivot(index="Dates", columns="commodity_label", values="signal_tanh").sort_index()
    pred_wide = predictions_long.pivot(index="Dates", columns="commodity_label", values="predicted_vol_scaled_21d_return").sort_index()

    # Ridge-specific names.
    predictions_long.to_csv(output_dir / "ridge_test_predictions_long.csv", index=False)
    signal_wide.to_csv(output_dir / "ridge_test_signals_wide.csv", index_label="Date")
    pred_wide.to_csv(output_dir / "ridge_test_predicted_returns_wide.csv", index_label="Date")
    metrics_df.to_csv(output_dir / "ridge_model_metrics.csv", index=False)
    ridge_coefficients.to_csv(output_dir / "ridge_coefficients.csv", index=False)

    with open(output_dir / "ridge_selected_features_by_commodity.json", "w") as f:
        json.dump(selected_feature_map, f, indent=2)

    # Compatibility copies for the existing NN portfolio block.
    predictions_long.to_csv(output_dir / "nn_test_predictions_long.csv", index=False)
    signal_wide.to_csv(output_dir / "nn_test_signals_wide.csv", index_label="Date")
    pred_wide.to_csv(output_dir / "nn_test_predicted_returns_wide.csv", index_label="Date")
    metrics_df.to_csv(output_dir / "nn_model_metrics.csv", index=False)

    print("\nSaved Ridge outputs to:", output_dir.resolve())
    print("- ridge_test_signals_wide.csv")
    print("- ridge_test_predictions_long.csv")
    print("- ridge_model_metrics.csv")
    print("- ridge_coefficients.csv")
    print("\nFor the portfolio block, use:")
    print(f"SIGNALS_FILE = Path('{output_dir}/nn_test_signals_wide.csv')")

    return predictions_long, metrics_df, selected_feature_map, ridge_coefficients


if __name__ == "__main__":
    predictions_long, metrics_df, selected_features, ridge_coefficients = run_task3_ridge_pipeline()
    print("\nRidge model metrics:")
    print(metrics_df.to_string(index=False))



Fitting Ridge for Gold (GOLD 100 OZ)...
  alpha=10000.000000, features=43, train=4196, test_corr=0.2077, directional_accuracy=0.586

Fitting Ridge for Silver (SILVER)...
  alpha=10000.000000, features=43, train=4196, test_corr=0.1058, directional_accuracy=0.545

Fitting Ridge for Brent_Crude (BRENT CRUDE)...
  alpha=10000.000000, features=43, train=4196, test_corr=0.2468, directional_accuracy=0.606

Fitting Ridge for Natural_Gas (NATURAL GAS)...
  alpha=10000.000000, features=43, train=4196, test_corr=0.1037, directional_accuracy=0.568

Saved Ridge outputs to: /Users/jwu/Downloads/Backtesting Project/StatisticalModel/task3_ridge_outputs
- ridge_test_signals_wide.csv
- ridge_test_predictions_long.csv
- ridge_model_metrics.csv
- ridge_coefficients.csv

For the portfolio block, use:
SIGNALS_FILE = Path('task3_ridge_outputs/nn_test_signals_wide.csv')

Ridge model metrics:
commodity_label commodity_name   model  n_train  n_test_rows  n_test_eval_rows  n_features  selected_alpha safe_train_